# Citiz — 6. Où investir à Nantes Métropole : liste priorisée sous budget (Gold)
Runtime Python. À exécuter après `acces_notebook` et `modele_notebook` ; aucun téléchargement Bronze/Silver.
**Objectif 3** : trois types d'équipement (station Citiz, arrêt de bus ajouté sur une ligne existante,
station de vélos), un coût par type (`config/costs.yaml`, scénario moyen, horizon 3 ans), et une liste
priorisée qui maximise la population **nouvellement** desservie, pour des paliers de budget de 50 k€ à 1 M€.
Deux méthodes : glouton (meilleur ratio gain / coût) et ILP de couverture maximale (PuLP + HiGHS).
**Usage potentiel** d'une station Citiz : probabilité du modèle (`modele_notebook`) au carreau du candidat,
rapportée à la moyenne des candidats Citiz ; 1 pour les arrêts de bus et les stations de vélos (le modèle ne
parle que de Citiz). Couverture à vol d'oiseau (400 m, V1 : borne haute).
Un centre de carreau est une piste à vérifier sur le terrain, pas une place de stationnement disponible.

In [ ]:
%pip install -q "deltalake>=1,<2" scipy requests "pulp[highs]==4.1.0" "highspy==1.15.1"

In [ ]:
SILVER = 'silver'
GOLD = 'gold'
SCHEMA = 'dbo'
LOCAL_ROOT = None
EPCI = '244400404'       # Nantes Métropole
RAYON_M = 400
HORIZON_ANS = 3
# Coûts HT, scénario « moyenne » de config/costs.yaml (sources S1-S7 dans docs/sources_hypotheses.md).
COUTS = {
 'citiz_station': {'libelle': 'Station Citiz (1 véhicule + 1 place réservée)', 'investissement': 22000 + 1500,
                   'exploitation_annuelle': 1700 + 2500 + 600,
                   'source': 'véhicule et aménagement : hypothèses à valider ; assurance-entretien [S4]'},
 'bus_stop': {'libelle': 'Arrêt de bus accessible sur une ligne existante', 'investissement': 12500 + 2200,
              'exploitation_annuelle': 500, 'source': 'quai et abribus [S3] ; entretien : hypothèse à valider'},
 'bike_station': {'libelle': 'Station de vélos en libre-service (15 vélos)', 'investissement': 35000,
                  'exploitation_annuelle': 15 * 2500, 'source': 'coût complet GART 2 000-3 000 €/vélo/an [S2]'},
}
# Surcoût de risque de vol (station Citiz) : assurance et gestion au scénario « haute » de costs.yaml.
EXPLOITATION_CITIZ_RISQUE = 2100 + 4000 + 600
SEUIL_VOL_POUR_MILLE = 10      # vols de véhicules / 1 000 habitants (seuil du Gold), commune ≥ 10 000 habitants
POP_MIN_TAUX_PUBLIE = 10000
BUS_DISTANCE_MAX_M = 800       # arrêt candidat : carreau entre 400 et 800 m de l'arrêt existant le plus proche
BUDGETS = list(range(50000, 1000001, 50000))
BUDGETS_ILP = [100000, 250000, 500000, 1000000]
EQUITES = [0.0, 1.0]
ILP_SECONDES = 60
CO2_T_PAR_STATION_AN = 10000 * 0.142 / 1000   # 10 000 km évités × 142 gCO2e/km (bornes basses [S1], [S6])
SEUIL_DIFFUSION = 20

In [ ]:
import os, re, json, uuid, datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
import requests
from deltalake import DeltaTable, write_deltalake
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
RUN_ID = str(uuid.uuid4())
STATUS = []

def nu():
    import notebookutils
    return notebookutils

def root(lakehouse):
    if LOCAL_ROOT:
        return f"{LOCAL_ROOT}/{lakehouse}", None
    info = nu().lakehouse.get(lakehouse)
    if not isinstance(info, dict):
        info = json.loads(json.dumps(info, default=lambda x: x.__dict__))
    props = info.get("properties", {})
    path = props.get("abfsPath")
    if not path:
        raise RuntimeError(f"abfsPath absent pour {lakehouse}; contrôler notebookutils.lakehouse.get")
    return path.rstrip("/"), {"bearer_token": nu().credentials.getToken("storage"), "use_fabric_endpoint":"true"}

def table_path(lakehouse, name):
    p,o = root(lakehouse)
    return f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}{name}",o

def read_table(lakehouse, name, required=False, run_id=None, columns=None):
    p,o = table_path(lakehouse,name)
    try:
        dt_=DeltaTable(p,storage_options=o)
        if columns: columns=[c for c in columns+['_run_id'] if c in {f.name for f in dt_.schema().fields}]
        t=dt_.to_pandas(columns=columns)
    except Exception as e:
        if required: raise RuntimeError(f"Table obligatoire illisible: {lakehouse}.{name}") from e
        print(f"Absente/illisible: {lakehouse}.{name}: {str(e)[:120]}")
        return pd.DataFrame()
    if run_id is not None:
        if '_run_id' not in t: return pd.DataFrame()
        t=t[t['_run_id']==run_id].copy()
    return t.drop(columns=['_run_id'],errors='ignore')

def write_table(lakehouse,name,df):
    df=df.reset_index(drop=True).copy()
    df['_run_id']=RUN_ID
    # Types explicites même pour colonnes entièrement nulles / tables vides.
    a=pa.Table.from_pandas(df,preserve_index=False)
    a=pa.Table.from_arrays([a[c].cast(pa.string()) if pa.types.is_null(a[c].type) else a[c]
                            for c in a.column_names],names=a.column_names)
    p,o=table_path(lakehouse,name)
    if LOCAL_ROOT: os.makedirs(os.path.dirname(p),exist_ok=True)
    write_deltalake(p,a,mode='overwrite',schema_mode='overwrite',storage_options=o)
    STATUS.append({'table':name,'lignes':len(df),'run_id':RUN_ID})
    print(f"{lakehouse}.{name}: {len(df):,} lignes")

def xyz(lon,lat):
    a,b=np.radians(np.asarray(lon,dtype=float)),np.radians(np.asarray(lat,dtype=float))
    return np.column_stack([np.cos(b)*np.cos(a),np.cos(b)*np.sin(a),np.sin(b)])

def chord(metres):
    return 2*np.sin(metres/(2*6371008.8))

def metres(ch):
    return 2*6371008.8*np.arcsin(np.clip(ch/2,0,1))

def coverage(q, refs, m):
    pairs=cKDTree(xyz(q.lon,q.lat)).sparse_distance_matrix(cKDTree(xyz(refs.lon,refs.lat)),chord(m),output_type='ndarray')
    return csr_matrix((np.ones(len(pairs),dtype=bool),(pairs['i'],pairs['j'])),shape=(len(q),len(refs)))

def diffusable(s):
    return (np.round(s/5)*5).where(s>=SEUIL_DIFFUSION)

## 1 — Optimiseur (repris tel quel de `src/optim/optimize.py`, testé dans `tests/test_optimize.py`)

In [ ]:
"""
Choix d'emplacements sous contrainte de budget : glouton et ILP de couverture maximale (PuLP + HiGHS).

Les deux méthodes partagent la même entrée (`Problem`) et la même sortie (`Result`).

Fonction objectif (identique pour les deux méthodes) :
    pour chaque type d'équipement t et chaque carreau i NON couvert par l'offre existante de type t,
    si au moins un candidat retenu j de type t couvre i :
        gain_i = population_i × (1 + equite × priorite_equite_i) × usage_rel_j
    où usage_rel_j = usage_predit_j / moyenne(usage_predit) et j est, parmi les candidats retenus
    qui couvrent i, celui d'usage maximal. Un carreau compte au plus une fois par type.

- Glouton : à chaque étape, le candidat de meilleur ratio gain marginal / coût qui tient dans le budget
  restant ; la couverture est mise à jour après chaque choix.
- ILP : x_j ∈ {0,1} (candidat retenu), z_ij ∈ {0,1} (carreau i crédité à j) ;
  max Σ z_ij·gain_ij  s.c.  z_ij ≤ x_j,  Σ_{j de type t} z_ij ≤ 1,  Σ cout_j·x_j ≤ budget.
"""

import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import pulp
import scipy.sparse as sp

OUTPUT_COLUMNS = ["rang", "id", "type", "nom", "x", "y", "cout", "pop_nouvellement_desservie",
                  "usage_predit", "cout_par_habitant", "budget_cumule", "gain_objectif"]


@dataclass
class Problem:
    coverage: sp.csr_matrix   # carreaux × candidats, True si le candidat couvre le carreau (≤ 5 min à pied)
    cells: pd.DataFrame       # population, priorite_equite ∈ [0,1], couvert_<type> (bool) pour chaque type candidat
    candidates: pd.DataFrame  # id, type, nom, cout, usage_predit, x, y

    def __post_init__(self):
        self.coverage = sp.csr_matrix(self.coverage, dtype=bool)
        n_cells, n_cand = self.coverage.shape
        if n_cells != len(self.cells) or n_cand != len(self.candidates):
            raise ValueError(f"matrice {self.coverage.shape} incohérente avec "
                             f"{len(self.cells)} carreaux / {len(self.candidates)} candidats")
        missing = {f"couvert_{t}" for t in self.candidates["type"].unique()} - set(self.cells.columns)
        if missing:
            raise ValueError(f"colonnes de couverture existante manquantes : {sorted(missing)}")
        self.cells = self.cells.reset_index(drop=True)
        self.candidates = self.candidates.reset_index(drop=True)


@dataclass
class Result:
    selection: pd.DataFrame                  # colonnes OUTPUT_COLUMNS, triées par rang
    summary: dict = field(default_factory=dict)


# ---------------------------------------------------------------------------------------------
def _weights(p: Problem, equity: float) -> np.ndarray:
    pop = p.cells["population"].to_numpy(float)
    prio = p.cells.get("priorite_equite", pd.Series(0.0, index=p.cells.index)).fillna(0).to_numpy(float)
    return pop * (1.0 + equity * prio)


def _usage_rel(p: Problem) -> np.ndarray:
    u = p.candidates["usage_predit"].to_numpy(float)
    m = u.mean() if len(u) and u.mean() > 0 else 1.0
    return np.clip(u / m, 1e-6, None)


def _uncovered(p: Problem, t: str) -> np.ndarray:
    return ~p.cells[f"couvert_{t}"].fillna(False).to_numpy(bool)


def evaluate(p: Problem, chosen: list[int], equity: float = 0.0) -> float:
    """Valeur de la fonction objectif pour un ensemble de candidats (indices)."""
    w, u, total = _weights(p, equity), _usage_rel(p), 0.0
    cov = p.coverage.tocsc()
    types = p.candidates["type"].to_numpy()
    for t in set(types[chosen]) if chosen else ():
        best = np.zeros(cov.shape[0])
        for j in (j for j in chosen if types[j] == t):
            rows = cov.indices[cov.indptr[j]:cov.indptr[j + 1]]
            best[rows] = np.maximum(best[rows], u[j])
        total += float((w * best * _uncovered(p, t)).sum())
    return total


def _order_and_format(p: Problem, chosen: list[int], equity: float) -> pd.DataFrame:
    """Range les candidats retenus par ratio gain marginal / coût et calcule les colonnes de sortie."""
    w, u = _weights(p, equity), _usage_rel(p)
    pop = p.cells["population"].to_numpy(float)
    cov = p.coverage.tocsc()
    cand = p.candidates
    types = cand["type"].to_numpy()
    cost = cand["cout"].to_numpy(float)
    covered = {t: ~_uncovered(p, t) for t in set(types)}
    remaining, rows, cum = list(chosen), [], 0.0
    while remaining:
        best_j, best_r, best_gain, best_new = None, -1.0, 0.0, None
        for j in remaining:
            idx = cov.indices[cov.indptr[j]:cov.indptr[j + 1]]
            new = idx[~covered[types[j]][idx]]
            gain = float(w[new].sum() * u[j])
            r = gain / cost[j] if cost[j] > 0 else np.inf
            if r > best_r:
                best_j, best_r, best_gain, best_new = j, r, gain, new
        remaining.remove(best_j)
        covered[types[best_j]][best_new] = True
        cum += cost[best_j]
        newpop = float(pop[best_new].sum())
        rows.append({"id": cand.at[best_j, "id"], "type": types[best_j], "nom": cand.get("nom", pd.Series("", index=cand.index))[best_j],
                     "x": cand.get("x", pd.Series(np.nan, index=cand.index))[best_j],
                     "y": cand.get("y", pd.Series(np.nan, index=cand.index))[best_j],
                     "cout": cost[best_j], "pop_nouvellement_desservie": newpop,
                     "usage_predit": float(cand.at[best_j, "usage_predit"]),
                     "cout_par_habitant": cost[best_j] / newpop if newpop > 0 else np.inf,
                     "budget_cumule": cum, "gain_objectif": best_gain})
    df = pd.DataFrame(rows, columns=OUTPUT_COLUMNS[1:])
    df.insert(0, "rang", np.arange(1, len(df) + 1))
    return df


def _summary(p: Problem, method: str, budget: float, equity: float, sel: pd.DataFrame, chosen, t0, status) -> dict:
    return {"methode": method, "budget": budget, "equite": equity, "statut": status,
            "n_retenus": len(sel), "cout_total": float(sel["cout"].sum()),
            "pop_nouvellement_desservie": float(sel["pop_nouvellement_desservie"].sum()),
            "objectif": evaluate(p, list(chosen), equity), "duree_s": round(time.perf_counter() - t0, 3)}


# ---------------------------------------------------------------------------------------------
def greedy(p: Problem, budget: float, equity: float = 0.0) -> Result:
    t0 = time.perf_counter()
    w, u = _weights(p, equity), _usage_rel(p)
    cov = p.coverage.tocsc()
    types = p.candidates["type"].to_numpy()
    cost = p.candidates["cout"].to_numpy(float)
    covered = {t: ~_uncovered(p, t) for t in set(types)}
    chosen, left, avail = [], float(budget), set(range(len(cost)))
    while True:
        best_j, best_r, best_new = None, 0.0, None
        for j in avail:
            if cost[j] > left:
                continue
            idx = cov.indices[cov.indptr[j]:cov.indptr[j + 1]]
            new = idx[~covered[types[j]][idx]]
            gain = float(w[new].sum() * u[j])
            r = gain / cost[j] if cost[j] > 0 else (np.inf if gain > 0 else 0.0)
            if r > best_r:
                best_j, best_r, best_new = j, r, new
        if best_j is None:  # plus aucun candidat finançable avec un gain positif
            break
        chosen.append(best_j)
        avail.discard(best_j)
        covered[types[best_j]][best_new] = True
        left -= cost[best_j]
    sel = _order_and_format(p, chosen, equity)
    return Result(sel, _summary(p, "glouton", budget, equity, sel, chosen, t0, "ok"))


def ilp(p: Problem, budget: float, equity: float = 0.0, time_limit: float = 60, gap: float = 1e-4) -> Result:
    t0 = time.perf_counter()
    w, u = _weights(p, equity), _usage_rel(p)
    cov = p.coverage.tocoo()
    types = p.candidates["type"].to_numpy()
    cost = p.candidates["cout"].to_numpy(float)
    unc = {t: _uncovered(p, t) for t in set(types)}
    # seules les paires (carreau non couvert, candidat) avec un gain positif portent une variable z
    pairs = [(i, j) for i, j in zip(cov.row, cov.col) if unc[types[j]][i] and w[i] > 0 and cost[j] <= budget]
    useful = sorted({j for _, j in pairs})

    m = pulp.LpProblem("couverture_maximale", pulp.LpMaximize)
    x = m.add_variable_dicts("x", useful, cat=pulp.LpBinary)
    z = m.add_variable_dicts("z", range(len(pairs)), lowBound=0, upBound=1)  # continu : entier à l'optimum
    # Pénalité infime sur le coût : sans elle, un candidat sans gain marginal peut être retenu
    # (objectif inchangé). Elle reste < 0,1 % du plus petit gain : la couverture optimale n'est pas modifiée.
    gains = [w[i] * u[j] for i, j in pairs]
    eps = 1e-3 * min(gains) / max(1.0, sum(cost[j] for j in useful)) if gains else 0.0
    m += pulp.lpSum(z[k] * g for k, g in enumerate(gains)) - pulp.lpSum(eps * cost[j] * x[j] for j in useful)
    m += pulp.lpSum(cost[j] * x[j] for j in useful) <= budget
    by_cell_type: dict[tuple, list] = {}
    for k, (i, j) in enumerate(pairs):
        m += z[k] <= x[j]
        by_cell_type.setdefault((i, types[j]), []).append(k)
    for ks in by_cell_type.values():
        if len(ks) > 1:
            m += pulp.lpSum(z[k] for k in ks) <= 1
    stats = m.solve(pulp.HiGHS(msg=False, timeLimit=time_limit, gapRel=gap)) if useful else None
    chosen = [j for j in useful if (x[j].value() or 0) > 0.5] if stats else []
    status = stats.status.name if stats else "rien_a_optimiser"
    sel = _order_and_format(p, chosen, equity)
    return Result(sel, _summary(p, "ILP", budget, equity, sel, chosen, t0, status))


def optimize(p: Problem, budget: float, equity: float = 0.0, method: str = "glouton",
             types: list[str] | None = None, **kw) -> Result:
    """Point d'entrée unique (application, scripts). `types` restreint les types d'équipement candidats."""
    if types is not None:
        keep = p.candidates["type"].isin(types).to_numpy()
        p = Problem(p.coverage[:, np.flatnonzero(keep)], p.cells, p.candidates[keep])
    if method in ("glouton", "greedy"):
        return greedy(p, budget, equity)
    if method in ("ilp", "ILP"):
        return ilp(p, budget, equity, **kw)
    raise ValueError(f"méthode inconnue : {method}")

## 2 — Zone, carreaux et offre existante
Communes de Nantes Métropole par l'API Découpage administratif. Les carreaux portent déjà la couverture
existante calculée par `acces_notebook` (Citiz, Marguerite, TC, vélos) sur toute l'offre, y compris
celle des communes voisines.

In [ ]:
run=read_table(SILVER,'pipeline_run',True)
RUN_ID=run.iloc[0]['run_id']
r=requests.get(f'https://geo.api.gouv.fr/epcis/{EPCI}/communes',params={'fields':'code,nom'},timeout=60); r.raise_for_status()
zone=pd.DataFrame(r.json()).rename(columns={'code':'commune_code','nom':'commune'})
if zone.empty: raise ValueError(f'Aucune commune pour l EPCI {EPCI}')
cells=read_table(GOLD,'gold_acces_carreau',True,RUN_ID)
if cells.empty: raise ValueError('gold_acces_carreau absent pour ce run : lancer acces_notebook')
cells=cells[cells.commune_code.isin(zone.commune_code)].reset_index(drop=True)
offer=read_table(GOLD,'gold_offre_desservie',True,RUN_ID,columns=['point_id','famille','operateur','nom','lat','lon','commune_code'])
pop=read_table(SILVER,'fact_population',True,RUN_ID)
crime=read_table(SILVER,'fact_delinquance',True,RUN_ID)
cars=read_table(SILVER,'fact_motorisation',True,RUN_ID)
print(f"{len(zone)} communes, {len(cells):,} carreaux, {cells.pop_calee.sum():,.0f} habitants")

## 3 — Candidats et coûts
- **Station Citiz** : centres des carreaux habités et arrêts de TC de la zone, à plus de 400 m de toute
  voiture partagée (Citiz, Marguerite).
- **Arrêt de bus** : centres des carreaux habités entre 400 et 800 m de l'arrêt le plus proche
  (raccourci : Silver n'a pas le tracé des lignes ; à vérifier sur le plan du réseau).
- **Station de vélos** : centres des carreaux habités à plus de 400 m de toute station de vélos.
Surcoût de risque de vol pour Citiz : commune ≥ 10 000 habitants et vols de véhicules > 10 / 1 000 hab.

In [ ]:
def nearest_m(q, refs):
    if refs.empty: return np.full(len(q),np.inf)
    d,_=cKDTree(xyz(refs.lon,refs.lat)).query(xyz(q.lon,q.lat),k=1)
    return metres(d)

car=offer[offer.famille.isin(['citiz','autopartage'])]
tc=offer[offer.famille.eq('tc')]
bike=offer[offer.famille.eq('velo')]
base=cells[['carreau_id','commune_code','lat','lon']].rename(columns={'carreau_id':'source_id'})
c_cz=pd.concat([base.assign(origine='carreau'),
                tc[tc.commune_code.isin(zone.commune_code)][['point_id','commune_code','lat','lon','nom']]
                  .rename(columns={'point_id':'source_id'}).assign(origine='arret_tc')],ignore_index=True)
c_cz=c_cz[nearest_m(c_cz,car)>RAYON_M].assign(type='citiz_station')
d_tc=nearest_m(base,tc)
c_bus=base[(d_tc>RAYON_M)&(d_tc<=BUS_DISTANCE_MAX_M)].assign(type='bus_stop',origine='carreau')
c_bike=base[nearest_m(base,bike)>RAYON_M].assign(type='bike_station',origine='carreau')
cand=pd.concat([c_cz,c_bus,c_bike],ignore_index=True)
cand['id']=cand.type+':'+cand.source_id.astype(str)
cand['nom']=cand.nom.fillna('Centre de carreau '+cand.source_id.astype(str))
# Risque de vol : dernier taux publié de la commune ; inconnu sous 10 000 habitants (pas de surcoût).
p23=pop[pop.annee==2023].set_index('commune_code').population
vol=(crime[crime.categorie.eq('vol_vehicule')&crime.taux_pour_mille.notna()].sort_values('annee')
     .drop_duplicates('commune_code',keep='last').set_index('commune_code').taux_pour_mille)
cand['vols_vehicule_pour_mille']=cand.commune_code.map(vol)
cand['risque_vol']=np.select([cand.commune_code.map(p23).fillna(0)<POP_MIN_TAUX_PUBLIE,
                              cand.vols_vehicule_pour_mille>SEUIL_VOL_POUR_MILLE],['inconnu','eleve'],default='modere')
expl=cand.type.map({k:v['exploitation_annuelle'] for k,v in COUTS.items()})
expl=expl.where(~(cand.type.eq('citiz_station')&cand.risque_vol.eq('eleve')),EXPLOITATION_CITIZ_RISQUE)
cand['cout']=cand.type.map({k:v['investissement'] for k,v in COUTS.items()})+HORIZON_ANS*expl
proba=read_table(GOLD,'gold_modele_proba_carreau',False,RUN_ID,columns=['lat','lon','proba_citiz'])
cand=cand.reset_index(drop=True)
cand['proba_citiz']=np.nan
if len(proba):
    _,i=cKDTree(xyz(proba.lon,proba.lat)).query(xyz(cand.lon,cand.lat),k=1)
    cand['proba_citiz']=proba.proba_citiz.to_numpy()[i]
else:
    print('gold_modele_proba_carreau absent : usage potentiel = 1 (lancer modele_notebook)')
cz=cand.type.eq('citiz_station')&cand.proba_citiz.notna()
cand['usage_predit']=1.0
cand.loc[cz,'usage_predit']=cand.proba_citiz[cz]/cand.proba_citiz[cz].mean()
assert cand.id.is_unique and cand.cout.gt(0).all()
print(cand.groupby('type').agg(candidats=('id','size'),cout=('cout','median')).to_string())

## 4 — Problème d'optimisation
Gain d'un carreau i couvert par un candidat retenu de type t, si i n'est pas déjà couvert par ce type :
`pop_calee_i × (1 + équité × priorité_i) × usage_rel_j`.
`priorité_i = 0,5 × (1 − min(options, 6) / 6) + 0,5 × min(part de ménages sans voiture / 50 %, 1)` ∈ [0, 1] :
peu d'options et beaucoup de ménages sans voiture (Silver n'a pas le niveau de vie au carreau).

In [ ]:
sans_voiture=cars.sort_values('annee').drop_duplicates('commune_code',keep='last').set_index('commune_code').part_sans_voiture
prio=(0.5*(1-cells.options_ponderees.clip(upper=6)/6)
      +0.5*(cells.commune_code.map(sans_voiture).fillna(0)/0.5).clip(upper=1))
P=pd.DataFrame({'population':cells.pop_calee,'priorite_equite':prio,
                'couvert_citiz_station':cells.couvert_autopartage,'couvert_bus_stop':cells.couvert_tc,
                'couvert_bike_station':cells.couvert_velo})
cov=coverage(cells,cand,RAYON_M)
problem=Problem(cov,P,cand.assign(x=cand.lon,y=cand.lat))
print(f'Matrice {cov.shape}, {cov.nnz:,} paires')

## 5 — Résolution par paliers de budget
Glouton pour chaque palier ; ILP (optimal ou meilleur trouvé en 60 s) pour 100 k€, 250 k€, 500 k€ et 1 M€.

In [ ]:
sel, res = [], []
# perimetre 'tous' : les trois types se disputent le budget ; sinon un seul type (glouton), pour répondre
# aussi à « où ajouter un arrêt de bus » et « où ajouter une station de vélos ».
# scenario 'population seule' : même calcul sans le modèle (usage = 1), pour voir ce que le modèle change.
problems={'avec modele':problem,'population seule':Problem(cov,P,cand.assign(x=cand.lon,y=cand.lat,usage_predit=1.0))}
runs=([(m,'glouton',b,e,'tous') for m in problems for b in BUDGETS for e in EQUITES]
      +[(m,'ILP',b,e,'tous') for m in problems for b in BUDGETS_ILP for e in EQUITES]
      +[('avec modele','glouton',b,e,t) for t in COUTS for b in BUDGETS for e in EQUITES])
for scenario,methode,b,e,perimetre in runs:
    types=None if perimetre=='tous' else [perimetre]
    out=optimize(problems[scenario],b,e,method=methode,types=types,**({'time_limit':ILP_SECONDES} if methode=='ILP' else {}))
    sel.append(out.selection.assign(methode=methode,budget=b,equite=e,perimetre=perimetre,scenario=scenario))
    res.append({**out.summary,'perimetre':perimetre,'scenario':scenario})
sel=pd.concat(sel,ignore_index=True).rename(columns={'x':'lon','y':'lat'})
res=pd.DataFrame(res)
info=cand.set_index('id')[['commune_code','origine','source_id','risque_vol']]
sel=sel.join(info,on='id').join(cand.set_index('id')[['proba_citiz']],on='id').merge(zone,on='commune_code',how='left')
sel['co2_evite_t_an']=np.where(sel.type.eq('citiz_station'),CO2_T_PAR_STATION_AN,0.0)
sel['pop_nouvellement_desservie']=diffusable(sel.pop_nouvellement_desservie)
sel['cout_par_habitant']=sel.cout/sel.pop_nouvellement_desservie
res['pop_nouvellement_desservie']=np.round(res.pop_nouvellement_desservie/5)*5
# Contrôles : budget respecté, l'ILP fait au moins aussi bien que le glouton au même palier.
assert (res.cout_total<=res.budget+1e-6).all()
both=res[res.perimetre.eq('tous')].pivot_table(index=['scenario','budget','equite'],columns='methode',values='objectif').dropna()
assert (both['ILP']>=both['glouton']*(1-1e-3)).all(), both
print(res[res.budget.isin(BUDGETS_ILP)&res.perimetre.eq('tous')][['scenario','methode','budget','equite','n_retenus','cout_total',
          'pop_nouvellement_desservie','statut','duree_s']].to_string(index=False))

## 6 — Écriture des tables Gold
`gold_selection` : une ligne par emplacement retenu, par méthode, palier de budget et scénario d'équité.
Dans Power BI : filtrer `methode`, `equite` et `budget` (paramètre de simulation) ; trier par `rang`.

In [ ]:
cand_pub=cand.merge(zone,on='commune_code',how='left')
cand_pub['pop_couvrable']=diffusable(pd.Series(cov.T.astype(float)@P.population.to_numpy(),index=cand.index))
couts=pd.DataFrame([{'type':k,**v,'horizon_ans':HORIZON_ANS,'cout_total':v['investissement']+HORIZON_ANS*v['exploitation_annuelle']}
                    for k,v in COUTS.items()])
write_table(GOLD,'gold_candidats_nantes',cand_pub.drop(columns=['x','y'],errors='ignore'))
write_table(GOLD,'gold_selection',sel)
write_table(GOLD,'gold_resume_optim',res)
write_table(GOLD,'gold_couts',couts)
print('Optimisation prête. Usage potentiel Citiz = probabilité du modèle, relative à la moyenne des candidats.')